In [2]:
import numpy as np
import pandas as pd
import plotly.express as px

In [41]:
# 1. Setup - Let's run 10,000 "what-if" scenarios
simulations = 10000

In [42]:
# Normal: Sales volume (Mean 1000 units, SD 150)
sales_vol = np.random.normal(loc= 1000,
                             scale= 150,
                             size= simulations)
sales_vol

array([1025.16935942, 1010.62920527,  970.99476404, ..., 1008.96878475,
        817.82295459,  998.29535593])

In [43]:
# Uniform: Production cost per unit ($5 to $9)
unit_cost = np.random.uniform(5, 9, simulations)

In [44]:
# Empirical: Sampling from a list of actual historical discount rates
hist_discounts = [0.0, 0.05, 0.10, 0.15, 0.20, 0.50]
probs = [0.7, 0.05, 0.05, 0.02, 0.08, 0.1]
discount_rate = np.random.choice(hist_discounts, simulations, p=probs)

In [45]:
# Weighted: Economy state (60% Stable, 30% Growth, 10% Recession)
states = ['Stable', 'Growth', 'Recession']
market_impact = np.random.choice([1.0, 1.3, 0.7], simulations, p=[0.6, 0.3, 0.1])

In [68]:
# 2. The Logic: Profit = Profit - Costs

# We'll assume a fixed base price of $15
base_price = 15

# Profit = (Volume * Price * Impact * (1-Discount))
revenue = (sales_vol * base_price * market_impact * (1 - discount_rate))

# Costs = (Volume * Cost)
total_costs = (sales_vol * unit_cost)

# Net
profit = revenue - total_costs

In [69]:
# 3. Quick Analysis
print(f"Average Profit: ${profit.mean():,.2f}")
print(f"Risk of loss (Profit < 0): {(profit < 0).mean() * 100:,.2f}%")

Average Profit: $7,674.84
Risk of loss (Profit < 0): 3.28%


In [53]:
# Dataframe
profit = pd.DataFrame(profit, columns=['Profit'])
profit['negative']= np.where(profit['Profit'] < 0, 1, 0)

#Plot
px.histogram(data_frame=profit,
             x= 'Profit',
             nbins=100,
             title='Distribution of Profits Simulations',
             width=800, height=500,
             color='negative',
             color_discrete_map={1:'red', 0:'green'}
             )

## Example 2

In [1]:
import numpy as np
import pandas as pd

# 1. Define our distribution parameters
iterations = 10000

# Normal: Sales volume (mean=100, std=15)
sales_vol = np.random.normal(100, 15, iterations)

# Uniform: Shipping cost per unit ($5 to $15)
shipping_cost = np.random.uniform(5, 15, iterations)

# Empirical: Historical returns (sampling directly from real data)
past_returns = [0.02, 0.05, 0.01, 0.10, 0.03, 0.08] # Small sample of actual data
returns_rate = np.random.choice(past_returns, iterations)

# Weighted: Market condition (40% Bull, 40% Neutral, 20% Bear)
conditions = ['Bull', 'Neutral', 'Bear']
weights = [0.4, 0.4, 0.2]
market_state = np.random.choice(conditions, iterations, p=weights)

# 2. Build the simulation logic
# Let's say base price is $50, but market state adds a multiplier
multipliers = {'Bull': 1.2, 'Neutral': 1.0, 'Bear': 0.8}
state_mults = np.array([multipliers[s] for s in market_state])

# Final calculation: Profit = (Vol * Price * State) - (Vol * Shipping)
profit = (sales_vol * 50 * state_mults) - (sales_vol * shipping_cost)

# 3. Analyze the chaos
results = pd.DataFrame({'Profit': profit})
print(f"Mean Profit: ${results['Profit'].mean():.2f}")
print(f"5th Percentile (Risk): ${results['Profit'].quantile(0.05):.2f}")
print(f"95th Percentile (Upside): ${results['Profit'].quantile(0.95):.2f}")

Mean Profit: $4199.40
5th Percentile (Risk): $2597.05
95th Percentile (Upside): $5933.65


## PERT
Most tasks have a "long tail". They rarely finish early, but they can certainly finish very late. That’s why we use the PERT (Program Evaluation and Review Technique) distribution.

It’s essentially a Beta distribution that's been reshaped to fit three simple guesses: your Optimistic, Most Likely, and Pessimistic timelines.

In [70]:
import numpy as np
import pandas as pd

def pert_sample(low, likely, high, size=10000):
    """Generates random samples using the PERT (Beta) distribution."""
    # Calculate the mean and standard deviation for the Beta shape
    mean = (low + 4 * likely + high) / 6
    a = 1 + 4 * (likely - low) / (high - low)
    b = 1 + 4 * (high - likely) / (high - low)
    # Scale the standard Beta to our specific range
    return np.random.beta(a, b, size) * (high - low) + low

# 1. Define Task Estimates (Low, Likely, High)
design = pert_sample(5, 7, 15)   # Often goes over
coding = pert_sample(10, 12, 20)  # High uncertainty
testing = pert_sample(3, 5, 8)    # More predictable

# 2. Total Project Duration
total_days = design + coding + testing

# 3. Decision Making
print(f"Average Duration: {total_days.mean():.1f} days")
print(f"90% Confidence Finish: {np.percentile(total_days, 90):.1f} days")

# What's the chance we finish in 25 days?
prob_25 = (total_days <= 25).mean() * 100
print(f"Probability of finishing in 25 days: {prob_25:.1f}%")

Average Duration: 26.2 days
90% Confidence Finish: 29.7 days
Probability of finishing in 25 days: 35.1%
